# Ethereum Smart Contract Vulnerability Detection — Cross-Modal Method Suite

**Dataset**: DIVE — 22,330 Etherscan-verified Ethereum contracts.
**Task**: Multi-label classification across **8 vulnerability classes** (not mutually exclusive):
`Reentrancy`, `Access Control`, `Arithmetic`, `Unchecked Return Values`, `DoS`,
`Bad Randomness`, `Front Running`, `Time manipulation`.
**Hardware**: Kaggle GPU **T4 × 2** (16 GB × 2 = 32 GB total VRAM).

## Research design — bytecode-only inference, multi-modal training

The deployed model **takes bytecode as its only input** — this is the realistic deployment setting, since on-chain contracts (especially unverified ones) only expose bytecode. **During training we are free to use source code** as a richer supervisory signal. This is the classical *Learning Using Privileged Information* (LUPI / Vapnik & Vashist 2009) setting; in modern deep-learning practice the dominant implementation is **cross-modal knowledge distillation** — train a strong teacher on source, freeze it, use its predictions as soft targets for a bytecode-only student.

### Three sections, three roles

| Section | Methods                                            | Modality at training | **Modality at inference** | Deployable? |
|---------|----------------------------------------------------|----------------------|---------------------------|-------------|
| **A — Bytecode-only baselines**             | B1, B2, B3, B4         | bytecode             | bytecode                  | ✅ yes      |
| **B — Source teachers** (training-only)     | S1, S2, S3, S4         | source code          | source code               | ❌ no (ceiling reference + soft-target source) |
| **C — Cross-modal distilled students**      | X1, X2, X3             | bytecode + soft labels from B's teachers | **bytecode**  | ✅ yes      |

Expected ordering on macro-F1:
**A (floor) ≤ C (cross-modal, bytecode-only) ≤ B (ceiling, needs source)**

The contribution is **C** — students that match deployment constraints (bytecode-only) yet recover some of the source-modality gap via distillation.

### The eight methods

**Section A — Bytecode-only baselines**
| ID | Method                                     | Class                |
|----|--------------------------------------------|----------------------|
| B1 | TF-IDF opcode 1+2-grams → Logistic Regression | classical          |
| B2 | TextCNN on opcode sequences                | deep / sequential    |
| B3 | BiLSTM + attention on opcode sequences     | deep / sequential    |
| B4 | Control-Flow-Graph + **GAT**               | deep / graph         |

**Section B — Source teachers** (trained, then *frozen* and used to soft-label all 22 330 contracts; never deployed)
| ID | Backbone                                  | Params  |
|----|-------------------------------------------|---------|
| S1 | `microsoft/codebert-base`                 | 125 M   |
| S2 | `microsoft/graphcodebert-base`            | 125 M   |
| S3 | `microsoft/unixcoder-base`                | 125 M   |
| S4 | `deepseek-ai/deepseek-coder-1.3b-base` + QLoRA | 1.3 B (swap to `-6.7b` if you have hours to spare) |

**Section C — Cross-modal distilled students** (all take bytecode only)
| ID | Architecture | Soft-target source |
|----|--------------|--------------------|
| X1 | TextCNN on opcodes | strongest small teacher (S3 by default) |
| X2 | TextCNN on opcodes | largest teacher (S4) |
| X3 | TextCNN on opcodes | mean ensemble of S1–S4 |

> **Note on `gemma-4-31B`** — that model name does not exist (Google's line-up tops at `gemma-3-27b` / `codegemma-7b`), and crucially, **31 B will not fit on T4 × 2 even with QLoRA**: 4-bit weights alone are ~16 GB versus 32 GB total VRAM, before activations and optimizer state. Practical QLoRA ceiling on T4 × 2 is ~7 B. To use Gemma, set `MODEL_NAME_S4 = "google/codegemma-7b"`.


## 1 — Environment setup

In [ ]:
# Kaggle has torch/transformers/lightgbm/sklearn. Install the rest.
!pip install -q iterative-stratification torch-geometric peft bitsandbytes accelerate -U


In [ ]:
import os, gc, re, json, math, time, random, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, roc_auc_score, hamming_loss,
                             average_precision_score)
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch: {torch.__version__} | CUDA: {torch.cuda.is_available()} "
      f"| GPUs: {torch.cuda.device_count()}")


## 2 — Config (edit `DATA_ROOT` if your Kaggle slug differs)

In [ ]:
DATA_ROOT = Path("/kaggle/input/datasets/henrychristian7555/dive-smart-contract-multi-class-vulnerability")
if not DATA_ROOT.exists():
    # Fallback: locate any directory containing DIVE_Labels.csv (handles slug variations).
    cand = next((p.parent for p in Path("/kaggle/input").rglob("DIVE_Labels.csv")), None)
    if cand is None:
        raise FileNotFoundError("Set DATA_ROOT manually — no DIVE_Labels.csv found under /kaggle/input/")
    DATA_ROOT = cand
    print(f"Auto-detected DATA_ROOT: {DATA_ROOT}")

SRC_DIR      = DATA_ROOT / "Source"
LABEL_CSV    = DATA_ROOT / "DIVE_Labels.csv"
BYTECODE_CSV = DATA_ROOT / "Bytecode.csv"

OUT_DIR        = Path("/kaggle/working"); OUT_DIR.mkdir(exist_ok=True, parents=True)
CACHE_DIR      = OUT_DIR / "cache";          CACHE_DIR.mkdir(exist_ok=True, parents=True)
TEACHER_PRED_DIR = OUT_DIR / "teacher_preds"; TEACHER_PRED_DIR.mkdir(exist_ok=True, parents=True)

LABEL_COLS = ["Reentrancy", "Access Control", "Arithmetic", "Unchecked Return Values",
              "DoS", "Bad Randomness", "Front Running", "Time manipulation"]
N_LABELS = len(LABEL_COLS)

# Master toggles — set RUN_S4 = False the first time if you want quick iteration.
RUN_S4 = True

assert SRC_DIR.exists() and LABEL_CSV.exists() and BYTECODE_CSV.exists(), \
       f"Missing expected files under {DATA_ROOT} — check the path."
print(f"DATA_ROOT: {DATA_ROOT}")
print("Sources OK.")


## 3 — Labels, distribution, and multi-label stratified split

In [ ]:
labels = pd.read_csv(LABEL_CSV)
print("Shape:", labels.shape)
pos = labels[LABEL_COLS].sum().sort_values(ascending=False)
prev = (pos / len(labels) * 100).round(2)
print(pd.DataFrame({"#pos": pos.astype(int), "prevalence_%": prev}))

ax = pos.plot(kind="bar", figsize=(9, 3.2), color="steelblue")
ax.set_title("DIVE label distribution"); ax.set_ylabel("# positive contracts"); plt.show()

card = labels[LABEL_COLS].sum(axis=1)
print(f"Label cardinality — mean: {card.mean():.2f}  max: {card.max()}  "
      f"zero-label: {(card==0).sum()} ({(card==0).mean()*100:.1f}%)")


In [ ]:
Y = labels[LABEL_COLS].values
mss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
idx_trv, idx_te = next(mss1.split(labels["contractID"].values, Y))
mss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
sub_tr, sub_v = next(mss2.split(labels.iloc[idx_trv]["contractID"].values, Y[idx_trv]))
idx_tr = idx_trv[sub_tr]; idx_v = idx_trv[sub_v]

split = np.empty(len(labels), dtype=object)
split[idx_tr] = "train"; split[idx_v] = "val"; split[idx_te] = "test"
labels["split"] = split

train_df = labels[labels.split=="train"].reset_index(drop=True)
val_df   = labels[labels.split=="val"].reset_index(drop=True)
test_df  = labels[labels.split=="test"].reset_index(drop=True)
Y_tr = train_df[LABEL_COLS].values.astype(np.float32)
Y_v  = val_df[LABEL_COLS].values.astype(np.float32)
Y_te = test_df[LABEL_COLS].values.astype(np.float32)
for s in ["train","val","test"]:
    print(f"  {s:5s}  n={int((labels.split==s).sum())}")


## 4 — Shared evaluator and bookkeeping

In [ ]:
def evaluate(y_true, y_prob, threshold=0.5, name="", deployable=None):
    # Multi-label metrics. y_prob is (n, n_labels) in [0,1].
    y_pred = (y_prob >= threshold).astype(int)
    out = {
        "name":         name,
        "deployable":   deployable,                # True / False / None
        "macro_f1":     f1_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_f1":     f1_score(y_true, y_pred, average="micro", zero_division=0),
        "weighted_f1":  f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
    }
    try:
        out["macro_auc"] = roc_auc_score(y_true, y_prob, average="macro")
        out["micro_auc"] = roc_auc_score(y_true, y_prob, average="micro")
        out["macro_ap"]  = average_precision_score(y_true, y_prob, average="macro")
    except ValueError:
        out["macro_auc"] = out["micro_auc"] = out["macro_ap"] = float("nan")
    for c, f in zip(LABEL_COLS, f1_score(y_true, y_pred, average=None, zero_division=0)):
        out[f"f1[{c}]"] = f
    return out

def make_pos_weight(Y, device):
    pos = Y.sum(axis=0); neg = len(Y) - pos
    pw = neg / np.clip(pos, 1, None)
    return torch.tensor(pw, dtype=torch.float32, device=device)

RESULTS = []


## 5 — Bytecode preprocessing (used by all B-methods and the C-students)

Disassemble each contract's runtime bytecode into an opcode mnemonic sequence and cache it.
Standard preprocessing: collapse `PUSH1..PUSH32` / `DUP1..DUP16` / `SWAP1..SWAP16` to `PUSH` / `DUP` / `SWAP` (ContractWard and successors).


In [ ]:
EVM_OPCODES = {
    0x00:"STOP",0x01:"ADD",0x02:"MUL",0x03:"SUB",0x04:"DIV",0x05:"SDIV",0x06:"MOD",
    0x07:"SMOD",0x08:"ADDMOD",0x09:"MULMOD",0x0a:"EXP",0x0b:"SIGNEXTEND",
    0x10:"LT",0x11:"GT",0x12:"SLT",0x13:"SGT",0x14:"EQ",0x15:"ISZERO",0x16:"AND",
    0x17:"OR",0x18:"XOR",0x19:"NOT",0x1a:"BYTE",0x1b:"SHL",0x1c:"SHR",0x1d:"SAR",
    0x20:"SHA3",
    0x30:"ADDRESS",0x31:"BALANCE",0x32:"ORIGIN",0x33:"CALLER",0x34:"CALLVALUE",
    0x35:"CALLDATALOAD",0x36:"CALLDATASIZE",0x37:"CALLDATACOPY",0x38:"CODESIZE",
    0x39:"CODECOPY",0x3a:"GASPRICE",0x3b:"EXTCODESIZE",0x3c:"EXTCODECOPY",
    0x3d:"RETURNDATASIZE",0x3e:"RETURNDATACOPY",0x3f:"EXTCODEHASH",
    0x40:"BLOCKHASH",0x41:"COINBASE",0x42:"TIMESTAMP",0x43:"NUMBER",
    0x44:"DIFFICULTY",0x45:"GASLIMIT",0x46:"CHAINID",0x47:"SELFBALANCE",0x48:"BASEFEE",
    0x50:"POP",0x51:"MLOAD",0x52:"MSTORE",0x53:"MSTORE8",0x54:"SLOAD",0x55:"SSTORE",
    0x56:"JUMP",0x57:"JUMPI",0x58:"PC",0x59:"MSIZE",0x5a:"GAS",0x5b:"JUMPDEST",
    0xf0:"CREATE",0xf1:"CALL",0xf2:"CALLCODE",0xf3:"RETURN",0xf4:"DELEGATECALL",
    0xf5:"CREATE2",0xfa:"STATICCALL",0xfd:"REVERT",0xfe:"INVALID",0xff:"SELFDESTRUCT",
}
for i in range(32): EVM_OPCODES[0x60+i] = f"PUSH{i+1}"
for i in range(16): EVM_OPCODES[0x80+i] = f"DUP{i+1}"
for i in range(16): EVM_OPCODES[0x90+i] = f"SWAP{i+1}"
for i in range(5):  EVM_OPCODES[0xa0+i] = f"LOG{i}"

def normalize_op(op):
    if op.startswith("PUSH"): return "PUSH"
    if op.startswith("DUP"):  return "DUP"
    if op.startswith("SWAP"): return "SWAP"
    return op

def disassemble(hex_str, max_ops=4096):
    if not isinstance(hex_str, str) or len(hex_str) < 4: return []
    if hex_str.startswith("0x"): hex_str = hex_str[2:]
    if len(hex_str) % 2: hex_str = hex_str[:-1]
    try: b = bytes.fromhex(hex_str)
    except ValueError: return []
    ops = []; i = 0; n = len(b)
    while i < n:
        op = b[i]; ops.append(EVM_OPCODES.get(op, "UNK"))
        i += 1 + (op - 0x5f) if 0x60 <= op <= 0x7f else 1
        if len(ops) >= max_ops: break
    return ops


In [ ]:
op_cache_path = CACHE_DIR / "opcode_strs.json"
if op_cache_path.exists():
    opcode_strs = {int(k): v for k, v in json.loads(op_cache_path.read_text()).items()}
    print(f"Loaded cached opcodes for {len(opcode_strs)} contracts")
else:
    opcode_strs = {}
    t0 = time.time()
    for chunk in pd.read_csv(BYTECODE_CSV, usecols=["contractID","bytecode"], chunksize=1000):
        for cid, code_hex in zip(chunk["contractID"], chunk["bytecode"]):
            ops = disassemble(code_hex, max_ops=4096)
            opcode_strs[int(cid)] = " ".join(normalize_op(o) for o in ops) if ops else ""
    op_cache_path.write_text(json.dumps(opcode_strs))
    print(f"Disassembled {len(opcode_strs)} contracts in {time.time()-t0:.1f}s; cached.")

def corpus_for(df_subset):
    return [opcode_strs.get(int(c), "") for c in df_subset["contractID"]]
X_op_tr = corpus_for(train_df); X_op_v = corpus_for(val_df); X_op_te = corpus_for(test_df)
print("Token counts (50/90/99 pctl):",
      np.percentile([len(s.split()) for s in X_op_tr], [50,90,99]).astype(int))


# Section A — Bytecode-only baselines

These methods see **only bytecode** at both training and inference. They establish the deployable floor: this is what you can achieve without source code.


## A · B1 — TF-IDF opcode n-grams + Logistic Regression

ContractWard-style classical baseline. TF-IDF over opcode 1- and 2-grams captures local
control-flow patterns (`SLOAD CALL` ↔ reentrancy, `TIMESTAMP LT` ↔ time manipulation),
then per-label balanced Logistic Regression handles class imbalance.


In [ ]:
vec = TfidfVectorizer(analyzer="word", token_pattern=r"\S+",
                       ngram_range=(1,2), min_df=5, max_df=0.95,
                       sublinear_tf=True, max_features=50000)
Xtr_tfidf = vec.fit_transform(X_op_tr)
Xv_tfidf  = vec.transform(X_op_v)
Xte_tfidf = vec.transform(X_op_te)
print("TF-IDF matrix:", Xtr_tfidf.shape)

test_probs_B1 = np.zeros_like(Y_te, dtype=float)
for i, lab in enumerate(LABEL_COLS):
    if Y_tr[:, i].sum() < 5: continue
    clf = LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced",
                             solver="liblinear", random_state=SEED)
    clf.fit(Xtr_tfidf, Y_tr[:, i])
    test_probs_B1[:, i] = clf.predict_proba(Xte_tfidf)[:, 1]

res = evaluate(Y_te, test_probs_B1, name="B1: TF-IDF + LR", deployable=True)
RESULTS.append(res)
print(json.dumps({k:(round(v,4) if isinstance(v,float) else v) for k,v in res.items()}, indent=2))


## A · B2 — TextCNN on opcode sequences

Embedding + parallel `Conv1d` filters (windows 3/4/5) + max-pool + linear head.
The same `TextCNN` architecture is also reused as the **student** in Section C —
keeping the architecture fixed isolates the effect of the soft-target signal.


In [ ]:
PAD, UNK = "<pad>", "<unk>"
op_counter = Counter()
for s in X_op_tr: op_counter.update(s.split())
vocab = [PAD, UNK] + [w for w,_ in op_counter.most_common()]
op2id = {w:i for i,w in enumerate(vocab)}
print("Opcode vocab:", len(vocab))

MAX_LEN = 2048
def encode_ops(s):
    ids = [op2id.get(t, 1) for t in s.split()[:MAX_LEN]]
    return ids + [0]*(MAX_LEN-len(ids))

class OpcodeSeqDS(Dataset):
    # If soft_targets is given, __getitem__ returns (x, y_hard, y_soft).
    def __init__(self, corpus, Y, soft_targets=None):
        self.X = np.array([encode_ops(s) for s in corpus], dtype=np.int64)
        self.Y = Y.astype(np.float32)
        self.S = soft_targets.astype(np.float32) if soft_targets is not None else None
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        if self.S is None: return self.X[i], self.Y[i]
        return self.X[i], self.Y[i], self.S[i]

ds_tr = OpcodeSeqDS(X_op_tr, Y_tr)
ds_v  = OpcodeSeqDS(X_op_v,  Y_v)
ds_te = OpcodeSeqDS(X_op_te, Y_te)

class TextCNN(nn.Module):
    def __init__(self, vocab_size, n_labels, emb=128, ch=128, ks=(3,4,5), drop=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(emb, ch, k, padding=k//2) for k in ks])
        self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(ch*len(ks), n_labels)
    def forward(self, x):
        e = self.emb(x).transpose(1,2)
        feats = [F.relu(c(e)).max(dim=2).values for c in self.convs]
        return self.fc(self.drop(torch.cat(feats, dim=1)))


In [ ]:
def train_seq_model(model, ds_tr_local, ds_v_local, ds_te_local,
                    epochs=6, batch=64, lr=1e-3, name="",
                    distill_alpha=None, deployable=True,
                    return_predictions_on=None):
    # Generic sequential model trainer.
    # distill_alpha: if not None, ds_tr_local must yield (x, y_hard, y_soft) and
    #   the loss is alpha*BCE(hard) + (1-alpha)*BCE(soft).
    # deployable: marks the eval result row as deployable (bytecode-only) or not.
    pw = make_pos_weight(Y_tr, device)
    bce_hard = nn.BCEWithLogitsLoss(pos_weight=pw)
    bce_soft = nn.BCEWithLogitsLoss()                        # soft targets already balanced

    dl_tr = DataLoader(ds_tr_local, batch_size=batch, shuffle=True,  num_workers=2, pin_memory=True)
    dl_v  = DataLoader(ds_v_local,  batch_size=batch*2, shuffle=False, num_workers=2, pin_memory=True)
    dl_te = DataLoader(ds_te_local, batch_size=batch*2, shuffle=False, num_workers=2, pin_memory=True)

    if torch.cuda.device_count() > 1: model = nn.DataParallel(model)
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    def step(batch_data, train):
        if distill_alpha is None:
            x, y = batch_data; x = x.to(device); y = y.to(device)
            if train: opt.zero_grad()
            logits = model(x)
            loss = bce_hard(logits, y)
        else:
            x, y, s = batch_data
            x = x.to(device); y = y.to(device); s = s.to(device)
            if train: opt.zero_grad()
            logits = model(x)
            loss = distill_alpha * bce_hard(logits, y) + (1 - distill_alpha) * bce_soft(logits, s)
        if train: loss.backward(); opt.step()
        return loss.item(), torch.sigmoid(logits).detach().cpu().numpy(), y.detach().cpu().numpy()

    def run(loader, train):
        model.train(train)
        L, P, T = [], [], []
        for b in loader:
            l, p, t = step(b, train); L.append(l); P.append(p); T.append(t)
        return np.mean(L), np.vstack(P), np.vstack(T)

    best_f1, best_state = -1, None
    for ep in range(1, epochs+1):
        t0 = time.time()
        tr_loss,_,_ = run(dl_tr, True)
        with torch.no_grad(): v_loss, v_p, v_y = run(dl_v, False)
        v_f1 = f1_score(v_y, (v_p>=0.5).astype(int), average="macro", zero_division=0)
        sched.step()
        print(f"  ep{ep}/{epochs}  tr={tr_loss:.4f} v={v_loss:.4f} v_macroF1={v_f1:.4f}  ({time.time()-t0:.1f}s)")
        if v_f1 > best_f1:
            best_f1 = v_f1
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best_state)
    with torch.no_grad(): _, te_p, te_y = run(dl_te, False)
    res = evaluate(te_y, te_p, name=name, deployable=deployable); RESULTS.append(res)
    print(json.dumps({k:(round(v,4) if isinstance(v,float) else v) for k,v in res.items()}, indent=2))
    del model; torch.cuda.empty_cache(); gc.collect()
    return res

train_seq_model(TextCNN(len(vocab), N_LABELS), ds_tr, ds_v, ds_te,
                epochs=6, batch=64, lr=1e-3, name="B2: TextCNN (opcodes)", deployable=True)


## A · B3 — BiLSTM + attention on opcode sequences

Captures dependencies spanning the whole trace — relevant because vulnerabilities like reentrancy connect an SLOAD early in the trace to an external CALL much later.


In [ ]:
class BiLSTMAttn(nn.Module):
    def __init__(self, vocab_size, n_labels, emb=128, hid=128, drop=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
        self.lstm = nn.LSTM(emb, hid, num_layers=1, batch_first=True, bidirectional=True)
        self.attn = nn.Linear(2*hid, 1)
        self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(2*hid, n_labels)
    def forward(self, x):
        mask = (x != 0).float().unsqueeze(-1)
        e = self.emb(x)
        h, _ = self.lstm(e)
        a = self.attn(h).masked_fill(mask == 0, -1e9)
        w = torch.softmax(a, dim=1)
        ctx = (w * h).sum(dim=1)
        return self.fc(self.drop(ctx))

train_seq_model(BiLSTMAttn(len(vocab), N_LABELS), ds_tr, ds_v, ds_te,
                epochs=6, batch=32, lr=1e-3, name="B3: BiLSTM+Attn (opcodes)", deployable=True)


## A · B4 — Control-Flow-Graph + Graph Attention Network

Each contract becomes a graph:
- **Nodes** = basic blocks (split at `JUMPDEST` and after terminators `JUMP/JUMPI/STOP/RETURN/REVERT/INVALID/SELFDESTRUCT`).
- **Node features** = normalized opcode histogram of the block.
- **Edges** = sequential fallthrough between consecutive blocks (bidirectional). Dynamic jumps require symbolic execution; we use the standard static-sequence approximation.

A 2-layer **GAT** + global mean-pool + linear head. Liu et al. (TKDE 2021) / Zhuang et al. (IJCAI 2020).


In [ ]:
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GATConv, global_mean_pool

OP_VOCAB = sorted({normalize_op(v) for v in EVM_OPCODES.values()} | {"UNK"})
OP_IDX = {o:i for i,o in enumerate(OP_VOCAB)}
N_OP = len(OP_VOCAB)
TERMINATORS = {"JUMP","JUMPI","STOP","RETURN","REVERT","INVALID","SELFDESTRUCT"}

def opcodes_to_graph(ops, max_blocks=300):
    blocks, cur = [], []
    for op in ops:
        if op == "JUMPDEST" and cur:
            blocks.append(cur); cur = [op]
        elif op in TERMINATORS:
            cur.append(op); blocks.append(cur); cur = []
        else:
            cur.append(op)
    if cur: blocks.append(cur)
    if not blocks: blocks = [["STOP"]]
    blocks = blocks[:max_blocks]
    X = np.zeros((len(blocks), N_OP), dtype=np.float32)
    for i, blk in enumerate(blocks):
        for op in blk:
            X[i, OP_IDX.get(op, OP_IDX["UNK"])] += 1
        s = X[i].sum()
        if s: X[i] /= s
    if len(blocks) >= 2:
        src = list(range(len(blocks)-1)) + list(range(1, len(blocks)))
        dst = list(range(1, len(blocks))) + list(range(len(blocks)-1))
    else:
        src, dst = [0], [0]
    return Data(x=torch.tensor(X), edge_index=torch.tensor([src, dst], dtype=torch.long))

def build_pyg(df_subset, Y_subset):
    items = []
    for (_, row), y in zip(df_subset.iterrows(), Y_subset):
        s = opcode_strs.get(int(row["contractID"]), "")
        d = opcodes_to_graph(s.split() if s else ["STOP"])
        d.y = torch.tensor(y, dtype=torch.float32).unsqueeze(0)
        items.append(d)
    return items

pyg_tr = build_pyg(train_df, Y_tr)
pyg_v  = build_pyg(val_df,   Y_v)
pyg_te = build_pyg(test_df,  Y_te)
nodes_per_graph = [d.x.size(0) for d in pyg_tr]
print(f"Built {len(pyg_tr)} train graphs; nodes/graph 50/90/99 pctl: "
      f"{np.percentile(nodes_per_graph,[50,90,99]).astype(int)}")

class GAT(nn.Module):
    def __init__(self, in_dim, n_labels, hid=64, heads=4, drop=0.3):
        super().__init__()
        self.g1 = GATConv(in_dim, hid, heads=heads, dropout=drop)
        self.g2 = GATConv(hid*heads, hid, heads=1, dropout=drop)
        self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(hid, n_labels)
    def forward(self, data):
        x, ei, batch = data.x, data.edge_index, data.batch
        x = F.elu(self.g1(x, ei)); x = F.elu(self.g2(x, ei))
        x = global_mean_pool(x, batch)
        return self.fc(self.drop(x))

EPOCHS_G, BATCH_G, LR_G = 10, 64, 5e-4
dl_tr_g = PyGDataLoader(pyg_tr, batch_size=BATCH_G, shuffle=True)
dl_v_g  = PyGDataLoader(pyg_v,  batch_size=BATCH_G*2)
dl_te_g = PyGDataLoader(pyg_te, batch_size=BATCH_G*2)
gat = GAT(N_OP, N_LABELS).to(device)
pw = make_pos_weight(Y_tr, device); crit_g = nn.BCEWithLogitsLoss(pos_weight=pw)
opt_g = torch.optim.AdamW(gat.parameters(), lr=LR_G, weight_decay=1e-5)
sched_g = torch.optim.lr_scheduler.CosineAnnealingLR(opt_g, T_max=EPOCHS_G)

def run_gat(loader, train):
    gat.train(train)
    L, P, Y = [], [], []
    for batch in loader:
        batch = batch.to(device)
        if train: opt_g.zero_grad()
        logits = gat(batch); loss = crit_g(logits, batch.y)
        if train: loss.backward(); opt_g.step()
        L.append(loss.item())
        P.append(torch.sigmoid(logits).detach().cpu().numpy())
        Y.append(batch.y.detach().cpu().numpy())
    return np.mean(L), np.vstack(P), np.vstack(Y)

best_f1, best_state = -1, None
for ep in range(1, EPOCHS_G+1):
    t0 = time.time()
    tr_loss,_,_ = run_gat(dl_tr_g, True)
    with torch.no_grad(): v_loss, v_p, v_y = run_gat(dl_v_g, False)
    v_f1 = f1_score(v_y, (v_p>=0.5).astype(int), average="macro", zero_division=0)
    sched_g.step()
    print(f"  ep{ep}/{EPOCHS_G}  tr={tr_loss:.4f} v={v_loss:.4f} v_macroF1={v_f1:.4f}  ({time.time()-t0:.1f}s)")
    if v_f1 > best_f1:
        best_f1 = v_f1; best_state = {k:v.detach().cpu().clone() for k,v in gat.state_dict().items()}
gat.load_state_dict(best_state)
with torch.no_grad(): _, te_p, te_y = run_gat(dl_te_g, False)
res = evaluate(te_y, te_p, name="B4: CFG + GAT", deployable=True); RESULTS.append(res)
print(json.dumps({k:(round(v,4) if isinstance(v,float) else v) for k,v in res.items()}, indent=2))
del gat; torch.cuda.empty_cache(); gc.collect()


# Section B — Source-code teachers (training-only)

These models see source code. They are **not** part of the deployed system — at inference
time the deployable artifact has bytecode-only input. We train them here for one purpose:
to produce **soft targets** that the Section-C students will distill from. We also report their
test-set scores as a *ceiling* reference (what we'd get if source were available at inference).

Each teacher writes its sigmoid probabilities on **all 22 330 contracts** to
`/kaggle/working/teacher_preds/<teacher>.npz` so the C-section can mix and match them.


## Source-code preprocessing

In [ ]:
_re_block = re.compile(r"/\*.*?\*/", re.DOTALL)
_re_line  = re.compile(r"//[^\n]*")

def read_sol(cid):
    p = SRC_DIR / f"{int(cid)}.sol"
    if not p.exists(): return ""
    try: return p.read_text(encoding="utf-8", errors="ignore")
    except Exception: return ""

def clean_sol(s):
    s = _re_block.sub(" ", s); s = _re_line.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip()

print("Sample (contract 1):", clean_sol(read_sol(1))[:200])


## Encoder teacher trainer — used by S1, S2, S3

Full-fine-tune trainer for 125 M-class RoBERTa-style code encoders (CodeBERT, GraphCodeBERT,
UniXcoder). Difference between the three: pre-training only — same data and same loop here, so any
quality gap attributes cleanly to the backbone. After training, the teacher runs inference
on *every* contract and saves probabilities for distillation.


In [ ]:
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

class SolEncoderDS(Dataset):
    def __init__(self, cids, Y, tokenizer, max_len=512):
        self.cids = list(cids); self.Y = Y.astype(np.float32)
        self.tok = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.cids)
    def __getitem__(self, i):
        enc = self.tok(clean_sol(read_sol(self.cids[i])),
                       truncation=True, max_length=self.max_len,
                       padding="max_length", return_tensors="pt")
        return (enc["input_ids"].squeeze(0), enc["attention_mask"].squeeze(0),
                torch.tensor(self.Y[i]))

class EncoderClassifier(nn.Module):
    def __init__(self, model_name, n_labels, drop=0.2):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.drop = nn.Dropout(drop)
        self.head = nn.Linear(self.backbone.config.hidden_size, n_labels)
    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        return self.head(self.drop(out.last_hidden_state[:, 0]))

def predict_probs(model, ds, batch):
    dl = DataLoader(ds, batch_size=batch, shuffle=False, num_workers=2, pin_memory=True)
    model.eval()
    probs = []
    with torch.no_grad():
        for ids, mask, _ in dl:
            ids = ids.to(device); mask = mask.to(device)
            with torch.cuda.amp.autocast():
                logits = model(ids, mask)
            probs.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.vstack(probs)

def finetune_encoder_teacher(model_name, name, save_key,
                              epochs=3, batch=16, lr=2e-5, max_len=512):
    print(f"\n=== {name}  [{model_name}] ===")
    tok = AutoTokenizer.from_pretrained(model_name)
    ds_tr_s = SolEncoderDS(train_df["contractID"], Y_tr, tok, max_len)
    ds_v_s  = SolEncoderDS(val_df["contractID"],   Y_v,  tok, max_len)
    ds_te_s = SolEncoderDS(test_df["contractID"],  Y_te, tok, max_len)

    dl_tr = DataLoader(ds_tr_s, batch_size=batch,   shuffle=True,  num_workers=2, pin_memory=True)
    dl_v  = DataLoader(ds_v_s,  batch_size=batch*2, shuffle=False, num_workers=2, pin_memory=True)

    model = EncoderClassifier(model_name, N_LABELS).to(device)
    if torch.cuda.device_count() > 1: model = nn.DataParallel(model)
    pw = make_pos_weight(Y_tr, device); crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total = len(dl_tr)*epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.1*total), total)
    scaler = torch.cuda.amp.GradScaler()

    def run(loader, train):
        model.train(train)
        L, P, Y = [], [], []
        for ids, mask, y in loader:
            ids = ids.to(device); mask = mask.to(device); y = y.to(device)
            if train: opt.zero_grad()
            with torch.cuda.amp.autocast():
                logits = model(ids, mask); loss = crit(logits, y)
            if train:
                scaler.scale(loss).backward()
                scaler.step(opt); scaler.update(); sched.step()
            L.append(loss.item())
            P.append(torch.sigmoid(logits.float()).detach().cpu().numpy())
            Y.append(y.detach().cpu().numpy())
        return np.mean(L), np.vstack(P), np.vstack(Y)

    best_f1, best_state = -1, None
    for ep in range(1, epochs+1):
        t0 = time.time()
        tr_loss,_,_ = run(dl_tr, True)
        with torch.no_grad(): v_loss, v_p, v_y = run(dl_v, False)
        v_f1 = f1_score(v_y, (v_p>=0.5).astype(int), average="macro", zero_division=0)
        print(f"  ep{ep}/{epochs}  tr={tr_loss:.4f} v={v_loss:.4f} v_macroF1={v_f1:.4f}  ({time.time()-t0:.1f}s)")
        if v_f1 > best_f1:
            best_f1 = v_f1
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best_state)

    # Predict on ALL splits — save for distillation use later
    p_tr = predict_probs(model, ds_tr_s, batch*2)
    p_v  = predict_probs(model, ds_v_s,  batch*2)
    p_te = predict_probs(model, ds_te_s, batch*2)
    np.savez(TEACHER_PRED_DIR / f"{save_key}.npz", train=p_tr, val=p_v, test=p_te)

    res = evaluate(Y_te, p_te, name=name, deployable=False); RESULTS.append(res)
    print(json.dumps({k:(round(v,4) if isinstance(v,float) else v) for k,v in res.items()}, indent=2))
    del model; torch.cuda.empty_cache(); gc.collect()
    return res

finetune_encoder_teacher("microsoft/codebert-base",      "S1: CodeBERT (teacher)",      "S1_codebert")
finetune_encoder_teacher("microsoft/graphcodebert-base", "S2: GraphCodeBERT (teacher)", "S2_graphcodebert")
finetune_encoder_teacher("microsoft/unixcoder-base",     "S3: UniXcoder (teacher)",     "S3_unixcoder")


## S4 — QLoRA on DeepSeek-Coder-1.3B (training-only teacher)

A ~10× larger backbone made trainable on a single T4 via:
- **4-bit NF4 quantization** (bitsandbytes) — 1.3 B fp16 weights (~2.6 GB) → ~0.7 GB.
- **LoRA adapters** on `q_proj`/`k_proj`/`v_proj`/`o_proj` — only ~0.1 % of params have gradients.
- **fp16 compute** — T4 has no bf16. Forced `bnb_4bit_compute_dtype=float16`.
- **Single-GPU placement** — 4-bit quantized layers can't be split via DataParallel; one T4 only.

Default model is `deepseek-coder-1.3b-base` (realistic ~1–2 h on T4). To upgrade, set
`MODEL_NAME_S4 = "deepseek-ai/deepseek-coder-6.7b-base"` or `"google/codegemma-7b"` — but
expect 6+ hours per epoch on T4. **Skip with `RUN_S4 = False` in the Config cell.**


In [ ]:
if not RUN_S4:
    print("Skipping S4 (set RUN_S4 = True to enable).")
else:
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              BitsAndBytesConfig, get_linear_schedule_with_warmup)
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

    MODEL_NAME_S4 = "deepseek-ai/deepseek-coder-1.3b-base"
    MAX_LEN_S4   = 1024
    BATCH_S4     = 2
    GRAD_ACCUM   = 8
    EPOCHS_S4    = 2
    LR_S4        = 2e-4

    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    tok_s4 = AutoTokenizer.from_pretrained(MODEL_NAME_S4)
    if tok_s4.pad_token is None: tok_s4.pad_token = tok_s4.eos_token

    model_s4 = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME_S4, num_labels=N_LABELS,
        problem_type="multi_label_classification",
        quantization_config=bnb_cfg, device_map={"": 0}, torch_dtype=torch.float16)
    model_s4.config.pad_token_id = tok_s4.pad_token_id
    model_s4 = prepare_model_for_kbit_training(model_s4)
    lora_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                          target_modules=["q_proj","k_proj","v_proj","o_proj"],
                          bias="none", task_type=TaskType.SEQ_CLS)
    model_s4 = get_peft_model(model_s4, lora_cfg)
    model_s4.print_trainable_parameters()

    class SolCausalDS(Dataset):
        def __init__(self, cids, Y, tokenizer, max_len=1024):
            self.cids = list(cids); self.Y = Y.astype(np.float32)
            self.tok = tokenizer; self.max_len = max_len
        def __len__(self): return len(self.cids)
        def __getitem__(self, i):
            enc = self.tok(clean_sol(read_sol(self.cids[i])),
                           truncation=True, max_length=self.max_len,
                           padding="max_length", return_tensors="pt")
            return (enc["input_ids"].squeeze(0), enc["attention_mask"].squeeze(0),
                    torch.tensor(self.Y[i]))

    ds_tr_s4 = SolCausalDS(train_df["contractID"], Y_tr, tok_s4, MAX_LEN_S4)
    ds_v_s4  = SolCausalDS(val_df["contractID"],   Y_v,  tok_s4, MAX_LEN_S4)
    ds_te_s4 = SolCausalDS(test_df["contractID"],  Y_te, tok_s4, MAX_LEN_S4)
    dl_tr_s4 = DataLoader(ds_tr_s4, batch_size=BATCH_S4,   shuffle=True,  num_workers=2, pin_memory=True)
    dl_v_s4  = DataLoader(ds_v_s4,  batch_size=BATCH_S4*2, shuffle=False, num_workers=2, pin_memory=True)
    dl_te_s4 = DataLoader(ds_te_s4, batch_size=BATCH_S4*2, shuffle=False, num_workers=2, pin_memory=True)

    pw_s4 = make_pos_weight(Y_tr, torch.device("cuda:0"))
    crit_s4 = nn.BCEWithLogitsLoss(pos_weight=pw_s4)
    opt_s4 = torch.optim.AdamW([p for p in model_s4.parameters() if p.requires_grad],
                                lr=LR_S4, weight_decay=0.0)
    total_s4 = (len(dl_tr_s4) // GRAD_ACCUM) * EPOCHS_S4
    sched_s4 = get_linear_schedule_with_warmup(opt_s4, int(0.05*total_s4), total_s4)

    def run_s4(loader, train):
        model_s4.train(train)
        L, P, Y = [], [], []
        opt_s4.zero_grad()
        for step, (ids, mask, y) in enumerate(loader):
            ids = ids.to("cuda:0"); mask = mask.to("cuda:0"); y = y.to("cuda:0")
            out = model_s4(input_ids=ids, attention_mask=mask)
            logits = out.logits.float(); loss = crit_s4(logits, y)
            if train:
                (loss / GRAD_ACCUM).backward()
                if (step + 1) % GRAD_ACCUM == 0:
                    opt_s4.step(); sched_s4.step(); opt_s4.zero_grad()
            L.append(loss.item())
            P.append(torch.sigmoid(logits).detach().cpu().numpy())
            Y.append(y.detach().cpu().numpy())
        return np.mean(L), np.vstack(P), np.vstack(Y)

    best_f1_s4, best_state_s4 = -1, None
    for ep in range(1, EPOCHS_S4+1):
        t0 = time.time()
        tr_loss,_,_ = run_s4(dl_tr_s4, True)
        with torch.no_grad(): v_loss, v_p, v_y = run_s4(dl_v_s4, False)
        v_f1 = f1_score(v_y, (v_p>=0.5).astype(int), average="macro", zero_division=0)
        print(f"  ep{ep}/{EPOCHS_S4}  tr={tr_loss:.4f} v={v_loss:.4f} v_macroF1={v_f1:.4f}  ({time.time()-t0:.1f}s)")
        if v_f1 > best_f1_s4:
            best_f1_s4 = v_f1
            best_state_s4 = {k:v.detach().cpu().clone()
                             for k,v in model_s4.state_dict().items() if "lora" in k.lower()}
    if best_state_s4 is not None:
        sd = model_s4.state_dict()
        for k, v in best_state_s4.items(): sd[k] = v.to(sd[k].device)
        model_s4.load_state_dict(sd)

    # Predict on all splits, save for distillation
    def predict_s4(ds):
        dl = DataLoader(ds, batch_size=BATCH_S4*2, shuffle=False, num_workers=2, pin_memory=True)
        model_s4.eval()
        probs = []
        with torch.no_grad():
            for ids, mask, _ in dl:
                ids = ids.to("cuda:0"); mask = mask.to("cuda:0")
                out = model_s4(input_ids=ids, attention_mask=mask)
                probs.append(torch.sigmoid(out.logits.float()).cpu().numpy())
        return np.vstack(probs)

    p_tr = predict_s4(ds_tr_s4); p_v = predict_s4(ds_v_s4); p_te = predict_s4(ds_te_s4)
    np.savez(TEACHER_PRED_DIR / "S4_deepseek.npz", train=p_tr, val=p_v, test=p_te)
    res = evaluate(Y_te, p_te, name="S4: DeepSeek-Coder-1.3B QLoRA (teacher)", deployable=False)
    RESULTS.append(res)
    print(json.dumps({k:(round(v,4) if isinstance(v,float) else v) for k,v in res.items()}, indent=2))
    del model_s4; torch.cuda.empty_cache(); gc.collect()


# Section C — Cross-modal distilled students (bytecode-only inference)

Each student is a **bytecode-only TextCNN** (same architecture as B2) trained with a combined loss:

$$\mathcal{L} = \alpha \cdot \mathrm{BCE}(\hat{y},\, y_{\text{hard}}) + (1-\alpha) \cdot \mathrm{BCE}(\hat{y},\, y_{\text{soft}})$$

- `y_hard` = ground-truth multi-label vector.
- `y_soft` = teacher's sigmoid probabilities on the same contract (computed in Section B from source code).
- `α = 0.5` by default — balanced hard supervision and soft distillation.

**The student never sees source code.** At inference time, the entire trained-teacher pipeline is discarded; only the student's bytecode forward pass runs.

Three variants:
- **X1** distills from the **strongest small encoder teacher** (S3 UniXcoder typically).
- **X2** distills from **S4 DeepSeek-Coder-1.3B QLoRA** (largest single teacher).
- **X3** distills from the **mean ensemble of S1–S4** — multi-teacher KD smooths the noise of any single teacher.


In [ ]:
def load_teacher(save_key):
    p = TEACHER_PRED_DIR / f"{save_key}.npz"
    if not p.exists():
        raise FileNotFoundError(f"Teacher preds not found: {p}. Run the corresponding Section-B cell first.")
    z = np.load(p)
    return z["train"], z["val"], z["test"]

def distill_textcnn(teacher_train_probs, name, epochs=8, batch=64, lr=1e-3, alpha=0.5):
    ds_tr_d = OpcodeSeqDS(X_op_tr, Y_tr, soft_targets=teacher_train_probs)
    return train_seq_model(TextCNN(len(vocab), N_LABELS),
                           ds_tr_d, ds_v, ds_te,
                           epochs=epochs, batch=batch, lr=lr,
                           name=name, distill_alpha=alpha, deployable=True)

# --- X1: distill from the best small-encoder teacher (we pick S3 UniXcoder by default) ---
s3_tr, s3_v, s3_te = load_teacher("S3_unixcoder")
distill_textcnn(s3_tr, name="X1: TextCNN ← UniXcoder (KD)")


In [ ]:
# --- X2: distill from S4 DeepSeek-Coder QLoRA ---
if RUN_S4 and (TEACHER_PRED_DIR / "S4_deepseek.npz").exists():
    s4_tr, _, _ = load_teacher("S4_deepseek")
    distill_textcnn(s4_tr, name="X2: TextCNN ← DeepSeek-Coder-1.3B (KD)")
else:
    print("Skipping X2 — S4 teacher not available.")


In [ ]:
# --- X3: multi-teacher KD (mean ensemble of available teachers) ---
teacher_files = sorted(p for p in TEACHER_PRED_DIR.glob("*.npz"))
print("Combining teachers:", [p.stem for p in teacher_files])

train_stack = np.stack([np.load(p)["train"] for p in teacher_files])  # (n_teachers, n_train, n_labels)
test_stack  = np.stack([np.load(p)["test"]  for p in teacher_files])
ens_train = train_stack.mean(axis=0)
ens_test  = test_stack.mean(axis=0)

# Also evaluate the raw mean-ensemble of teacher predictions as a reference
# (it's NOT deployable — uses source — but useful to see what the student is chasing).
res = evaluate(Y_te, ens_test,
               name=f"Mean-ensemble of {len(teacher_files)} source teachers (ref, NOT deployable)",
               deployable=False)
RESULTS.append(res)
print("Ensemble (ceiling-ish):", {k:(round(v,4) if isinstance(v,float) else v)
                                   for k,v in res.items() if k in ("macro_f1","micro_f1","macro_auc")})

distill_textcnn(ens_train, name="X3: TextCNN ← Mean(S1–S4) (multi-teacher KD)")


## 8 — Results comparison

In [ ]:
results_df = pd.DataFrame(RESULTS)
summary = ["name","deployable","macro_f1","micro_f1","weighted_f1","macro_auc","macro_ap","hamming_loss"]
print(results_df[summary].to_string(index=False))

per_class = results_df.set_index("name")[[f"f1[{c}]" for c in LABEL_COLS]]
per_class.columns = LABEL_COLS
print("\nPer-class F1:")
print(per_class.round(3))

results_df.to_csv(OUT_DIR / "method_results.csv", index=False)
print(f"\nSaved {OUT_DIR/'method_results.csv'}")


## Reading the table — the three-tier story

The methodology is built around three tiers, and the table should read top-to-bottom like this:

1. **Floor (deployable, bytecode-only)** — B1, B2, B3, B4. These are what you can do **without source code**, ever. The deployable artifact has bytecode as input and is realistic for unverified on-chain contracts.

2. **Ceiling (NOT deployable, needs source)** — S1, S2, S3, S4, and the multi-teacher ensemble. These tell you the upper bound of *information available in source*, assuming you had the contract source at inference. **You cannot ship these** in a bytecode-only deployment setting, but they bound the achievable headroom.

3. **Cross-modal students (deployable, bytecode-only)** — X1, X2, X3. These are the contribution. They take **bytecode only at inference**, but were trained against soft labels produced by the source-only teachers. The expected ordering is **B-floor ≤ X ≤ S-ceiling**, with the gap between X and B quantifying the value of cross-modal training.

If the *gap closure* — i.e. `(X_best − B_best) / (S_best − B_best)` — is large (say > 50 %), distillation is highly effective on this dataset. If it's small (< 20 %), the bytecode modality has limited capacity to absorb the teacher's signal and more powerful student architectures (e.g., transformer over opcodes) are the next move.

## Reasonable next steps

1. **Stronger student** — replace `TextCNN` with a small **Transformer over opcodes** (4–6 layers, 256 dim). Still bytecode-only at inference; usually 1–3 macro-F1 points over TextCNN as a distillation target.
2. **Per-label distillation weights** — replace flat `α = 0.5` with per-label weights chosen by validation. Some classes (e.g., `Front Running`) benefit more from soft targets than others.
3. **Temperature** — apply a temperature `T > 1` to teacher logits before sigmoid to smooth the distribution; classic KD adds `T²` scaling on the soft term.
4. **Feature distillation** — match the student's penultimate-layer features to a projection of the teacher's `[CLS]` embedding (Hinton-style logit KD + Romero-style FitNets). Often beats logit-only KD when teacher and student diverge in capacity.
5. **Out-of-fold teacher predictions** — current teacher train-set probs are computed on the same data the teacher trained on, so they're slightly overconfident. K-fold teacher training (e.g., 5-fold) yields cleaner soft targets at the cost of 5× teacher training time.
6. **Deployment regression test** — sanity-check that the final saved student has *zero* source-code dependency in its forward pass (no `read_sol`, no tokenizer for `.sol`). A simple unit test that loads the student weights and predicts purely from a bytecode hex string is the right gate before shipping.
